In [ ]:
import requests
import json

def launch_binder_and_get_jupyter_url(binder_url, connect_timeout=10):
    """
    Launch a BinderHub build using requests with streaming enabled.

    Returns:
        (True, jupyter_url_with_token) on success
        (False, error_message) on failure

    Notes:
    - This function blocks until BinderHub reports either success or failure.
    - No exceptions are raised; all errors are converted to return values.
    """

    # Step 1: initiate the HTTP request (may block until headers are received)
    try:
        resp = requests.get(
            binder_url,
            stream=True,
            headers={"Accept": "text/event-stream"},
            timeout=(connect_timeout, None),  # timeout only for connection, not for streaming
        )
    except Exception as e:
        return False, f"Failed to connect to BinderHub: {e}"

    # Step 2: validate HTTP response
    if resp.status_code != 200:
        return False, f"BinderHub returned HTTP {resp.status_code}"

    # Step 3: read Server-Sent Events (SSE) line by line
    try:
        for raw_line in resp.iter_lines(decode_unicode=True):
            if not raw_line:
                continue

            # BinderHub sends events in SSE format: "data: {...}"
            if not raw_line.startswith("data:"):
                continue

            data = raw_line[5:].strip()

            # Parse JSON payload
            try:
                event = json.loads(data)
            except json.JSONDecodeError:
                continue

            phase = event.get("phase")

            # ---- Optional: real-time logging ----
            print("EVENT:", event, flush=True)

            # Build failed
            if phase == "failed":
                message = event.get("message") or "Binder build failed"
                return False, message

            # Build succeeded and Jupyter server is ready
            if phase == "ready":
                url = event.get("url")
                token = event.get("token")

                if not url:
                    return False, "Binder reported ready phase without a URL"

                # Some BinderHub versions include the token separately,
                # others already embed it in the URL
                # if token and "token=" not in url:
                #     separator = "&" if "?" in url else "?"
                #     url = f"{url}{separator}token={token}"

                return True, {"url": url, "token": token}

        # Stream ended without receiving a final "ready" or "failed" event
        return False, "Binder event stream ended without a final result"

    except Exception as e:
        return False, f"Error while reading Binder event stream: {e}"


In [7]:
binder_base_url = "https://binder.intel4coro.de/build/gh"
repo_path = "yxzhan/cram-vrb-lab/dev"

launch_url = f"{binder_base_url}/{repo_path}"

ok, result = launch_binder_and_get_jupyter_url(launch_url)

if ok:
    print("JupyterLab URL:", result)
else:
    print("Binder failed:", result)

EVENT: {'phase': 'waiting', 'message': 'Waiting for build to start...\n'}
EVENT: {'message': 'Picked Git content provider.\n'}
EVENT: {'message': "Cloning into '/tmp/repo2dockerm4oddojs'...\n", 'phase': 'fetching'}
EVENT: {'message': 'Updating files:  10% (185/1691)\r', 'phase': 'fetching'}
EVENT: {'message': 'Updating files:  11% (187/1691)\r', 'phase': 'fetching'}
EVENT: {'message': 'Updating files:  12% (203/1691)\r', 'phase': 'fetching'}
EVENT: {'message': 'Updating files:  13% (220/1691)\r', 'phase': 'fetching'}
EVENT: {'message': 'Updating files:  14% (237/1691)\r', 'phase': 'fetching'}
EVENT: {'message': 'Updating files:  15% (254/1691)\r', 'phase': 'fetching'}
EVENT: {'message': 'Updating files:  16% (271/1691)\r', 'phase': 'fetching'}
EVENT: {'message': 'Updating files:  17% (288/1691)\r', 'phase': 'fetching'}
EVENT: {'message': 'Updating files:  18% (305/1691)\r', 'phase': 'fetching'}
EVENT: {'message': 'Updating files:  19% (322/1691)\r', 'phase': 'fetching'}
EVENT: {'messag

In [8]:
binder_base_url = "https://binder.dev.intel4coro.de/build/gh"
repo_path = "yxzhan/cram-vrb-lab/dev"

launch_url = f"{binder_base_url}/{repo_path}"

ok, result = launch_binder_and_get_jupyter_url(launch_url)

if ok:
    print("JupyterLab URL:", result['url'] + 'proxy/8899/?token' + result['token'])
else:
    print("Binder failed:", result)

EVENT: {'phase': 'built', 'imageName': 'intel4coro/yxzhan-2dcram-2dvrb-2dlab-eb909a:5fa052c84245330e614fa0443f55419d766b616e', 'message': 'Found built image, launching...\n'}
EVENT: {'phase': 'launching', 'message': 'Launching server...\n'}
EVENT: {'phase': 'launching', 'message': 'Server requested\n'}
EVENT: {'phase': 'launching', 'message': '2026-08-14T14:02:37Z [Normal] Successfully assigned binder/jupyter-yxzhan-cram-vrb-lab-tg7j1t4v to gpu-worker\n'}
EVENT: {'phase': 'launching', 'message': '2026-08-14T14:02:37Z [Normal] Pulling image "intel4coro/yxzhan-2dcram-2dvrb-2dlab-eb909a:5fa052c84245330e614fa0443f55419d766b616e"\n'}
EVENT: {'phase': 'launching', 'message': '2026-08-14T14:02:40Z [Normal] Successfully pulled image "intel4coro/yxzhan-2dcram-2dvrb-2dlab-eb909a:5fa052c84245330e614fa0443f55419d766b616e" in 2.637s (2.637s including waiting). Image size: 16413194694 bytes.\n'}
EVENT: {'phase': 'launching', 'message': '2026-08-14T14:02:40Z [Normal] Created container: notebook\n'}
E

In [3]:
baseurl = result["url"]
token = result["token"]

In [4]:
session = requests.Session()
session.get(f"{baseurl}?token={token}")

<Response [200]>

In [5]:
print("Status code:", resp.status_code)
for k, v in resp.headers.items():
    print(f"{k}: {v}")

print(resp.text)

NameError: name 'resp' is not defined